### Advanced Model ChatBundestag

Advanced model identifies all suitable markers for metadata retrieval in user query

In [49]:
import os
import json

# data handling & viz
import pandas as pd
import matplotlib.pyplot as plt

# language preprocessing
import re #regex
from wordcloud import WordCloud
import spacy # DE stopwords


# langchain packages
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.faiss import DistanceStrategy

from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.embeddings import Embeddings

from langchain_groq import ChatGroq
from langchain_classic import hub


from pydantic import BaseModel, ValidationError
from typing import Optional, Literal, List


# environment variables
from dotenv import load_dotenv
import warnings

load_dotenv()
warnings.filterwarnings('ignore')

# -- LLM -- instantiate ChatGroq with llama
llm = ChatGroq(
    model= "llama-3.3-70b-versatile", #before: llama-3.1-8b-instant
    temperature=0,
    max_tokens=4096, # was None — explicit limit prevents silent truncation
    timeout=None,
    max_retries=2
)


# -- Data -- refs for vector db handling
vector_db_name = "debates_lp19"
vector_db_path = f"vector_databases/vector_db_{vector_db_name}"


In [2]:
# cleaned and reduced data set
df_exp_debates = pd.read_csv("data/debates_lp19.csv")

In [3]:
df_exp_debates.shape 

(26902, 11)

In [4]:
# get array of unique speakers in df 
KNOWN_SPEAKERS = {
    name.strip() 
    for name in df_exp_debates['speech_identification_ent'].dropna().unique() 
    if len(name.strip()) > 1 #strip empty rows
}

### Chunking

In [5]:
# chunk size based on document size
TINY_DOC_THRESHOLD   = 50    # e.g. interjections
SMALL_DOC_THRESHOLD  = 400   # short statements = 1 chunk
MEDIUM_DOC_THRESHOLD = 1500  # medium speeches = paragraph-level split
# anything above: full recursive split with overlap

MIN_CHUNK_CHARS   = 100      # drop anything shorter than this
CHUNK_SIZE_MEDIUM = 400      # characters #was 500
CHUNK_SIZE_LARGE  = 500      # characters #was 400
CHUNK_OVERLAP     = 50       # roughly 10%

BATCH_SIZE = 500             # reduce to 250 if RAM issues


In [6]:
# split text
def get_splitter(chunk_size):
    return RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", " "],        
        
    )

paragraph_splitter = get_splitter(CHUNK_SIZE_MEDIUM)
token_splitter     = get_splitter(CHUNK_SIZE_LARGE)


In [7]:
# dynamic chunking (document-aware)
def chunk_single_document(doc: Document, doc_idx: int) -> list[Document]:
    """
    Applies dynamic chunking based on document length.
    Never merges across documents.

    Size buckets (characters):
        tiny   < 50     : 1 chunk, flagged (low semantic signal)
        small  < 400    : 1 chunk
        medium < 1500   : paragraph-level split
        large  ≥ 1500   : recursive token-level split with overlap
    """
    text = doc.page_content.strip()

    if not text:
        return []

    length = len(text)

    # Determine strategy
    if length < TINY_DOC_THRESHOLD:
        strategy = "tiny"
        splits = [doc]

    elif length < SMALL_DOC_THRESHOLD:
        strategy = "small"
        splits = [doc]

    elif length < MEDIUM_DOC_THRESHOLD:
        strategy = "medium"
        splits = paragraph_splitter.split_documents([doc])

    else:
        strategy = "large"
        splits = token_splitter.split_documents([doc])
        
    # filter out fragments below minimum length
    splits = [s for s in splits if len(s.page_content.strip()) >= MIN_CHUNK_CHARS]

    # attach metadata to every chunk
    total = len(splits)
    for i, chunk in enumerate(splits):
        chunk.metadata.update({
            "chunk_id":          f"doc{doc_idx}_chunk{i}",
            "chunk_index":       i,
            "total_chunks":      total,
            "is_full_document":  total == 1,
            "chunking_strategy": strategy,
            "doc_char_length":   length,
        })

    return splits


In [8]:
# batch pipeline
def df_to_document_batches(df, batch_size=BATCH_SIZE):
    """Generator: yields batches of Documents without loading full df into memory."""
    for start in range(0, len(df), batch_size):
        batch = df.iloc[start:start + batch_size]
        yield [
            Document(
                page_content=row['text'],
                metadata={
                    'row_index':        i, 
                    'speaker_name':     row['speech_identification_ent'],
                    'date':             row['date'],
                    'year': str(pd.to_datetime(row['date']).year), # new column with year from date column
                    'legislative_period': row['period'],
                    'session':          row['session'],
                    'role':             row['Role'],
                    'government_status': row['governing_Party'], # more precise
                    'party':            row['Party'],
                }
            )
            for i, row in batch.iterrows()
        ]

In [9]:
# run chunking pipeline
def run_chunking_pipeline(df, batch_size=BATCH_SIZE) -> list[Document]:
    """
    Full batched chunking pipeline.
    Tracks strategy distribution so you can inspect and tune thresholds.
    """
    all_chunks = []
    strategy_counts = {"tiny": 0, "small": 0, "medium": 0, "large": 0}
    total_docs = 0

    for batch_idx, doc_batch in enumerate(df_to_document_batches(df, batch_size)):
        for doc in doc_batch:
            doc_idx = doc.metadata['row_index']  # use DataFrame index 
            chunks = chunk_single_document(doc, doc_idx)
            all_chunks.extend(chunks)

            if chunks:
                strategy_counts[chunks[0].metadata["chunking_strategy"]] += 1

        total_docs += len(doc_batch)
        print(f"Batch {batch_idx + 1}: {total_docs} docs → {len(all_chunks)} chunks")

    print(f"\nChunking-Strategie: {strategy_counts}")
    print(f"Gesamt: {len(all_chunks)} Chunks aus {total_docs} Dokumenten")
    return all_chunks

In [10]:
# chunking
chunks = run_chunking_pipeline(df_exp_debates)

Batch 1: 500 docs → 3864 chunks
Batch 2: 1000 docs → 7674 chunks
Batch 3: 1500 docs → 11489 chunks
Batch 4: 2000 docs → 15743 chunks
Batch 5: 2500 docs → 20457 chunks
Batch 6: 3000 docs → 25116 chunks
Batch 7: 3500 docs → 28878 chunks
Batch 8: 4000 docs → 33737 chunks
Batch 9: 4500 docs → 38774 chunks
Batch 10: 5000 docs → 42732 chunks
Batch 11: 5500 docs → 46707 chunks
Batch 12: 6000 docs → 51781 chunks
Batch 13: 6500 docs → 55680 chunks
Batch 14: 7000 docs → 59600 chunks
Batch 15: 7500 docs → 63528 chunks
Batch 16: 8000 docs → 67599 chunks
Batch 17: 8500 docs → 71238 chunks
Batch 18: 9000 docs → 75369 chunks
Batch 19: 9500 docs → 79066 chunks
Batch 20: 10000 docs → 82929 chunks
Batch 21: 10500 docs → 86981 chunks
Batch 22: 11000 docs → 91091 chunks
Batch 23: 11500 docs → 96153 chunks
Batch 24: 12000 docs → 100128 chunks
Batch 25: 12500 docs → 104149 chunks
Batch 26: 13000 docs → 108100 chunks
Batch 27: 13500 docs → 113211 chunks
Batch 28: 14000 docs → 117011 chunks
Batch 29: 14500 do

### Embedding

In [11]:
# instantiate embedding model 
embedding = HuggingFaceEmbeddings(
        model_name='intfloat/multilingual-e5-small', 
        # cloud upgrade path: intfloat/multilingual-e5-base or deepset/gbert-base or BAAI/bge-m3
        model_kwargs={'device': 'mps'}, 
        # cloud: 'mps' -> 'cuda'.
        encode_kwargs={"normalize_embeddings": True,
                      "prompt": "passage: "  # adds prefix automatically at indexing time
                      }
    )

In [12]:
# prefix handling for embedding model
class E5QueryWrapper(Embeddings):
    """
    Wraps HuggingFaceEmbeddings to add the required 'query: ' prefix
    for multilingual-e5 models at retrieval time only.
    """
    def __init__(self, base_embedding: Embeddings):
        self.base = base_embedding

    def embed_query(self, text: str) -> List[float]:
        return self.base.embed_query(f"query: {text}")

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        # prefix 'passage: ' already handled in encode_kwargs at indexing time
        return self.base.embed_documents(texts)

embedding_wrapped = E5QueryWrapper(embedding)

In [13]:
# function to create and save vector store 
def create_and_store(chunks,vector_db_path,embedding):
    """
    this function creates a vector store from chunks and saves it locally
    """
    # create the vector store 
    vectorstore = FAISS.from_documents(
        documents=chunks,
        embedding=embedding, # parameter name is singular!
        distance_strategy=DistanceStrategy.COSINE  # or DistanceStrategy.DOT or DistanceStrategy.L2 
    )
    
     # save vector store locally
    vectorstore.save_local(vector_db_path)

    return vectorstore

In [14]:
# implement retrieval from FAISS db

def retrieve_from_vector_db(vector_db_path,embedding):
    """
    this function splits out a retriever object from a local vector store
    """
    
    vectorstore = FAISS.load_local(
        folder_path=vector_db_path,
        embeddings=embedding, # parameter name is plural!
        allow_dangerous_deserialization=True,
        distance_strategy=DistanceStrategy.COSINE  # or DistanceStrategy.DOT or DistanceStrategy.L2 
    )
    retriever = vectorstore.as_retriever(
        search_kwargs={
            'k':15, # k nearest
        }  
    )
    
    return retriever,vectorstore


In [15]:
# check if vector store exists. if no: creates vector store
if not os.path.exists(vector_db_path):
        print("Vector DB not found. Creating and embedding chunks.")
        vectorstore=create_and_store(chunks=chunks, vector_db_path=vector_db_path, embedding=embedding_wrapped)
        print(f"Vector DB save to {vector_db_path}")
else:
    print(f"Vector DB found at {vector_db_path}. Skipping embedding.")

Vector DB found at vector_databases/vector_db_debates_lp19. Skipping embedding.


In [16]:
# load the retriever and index
retriever,vectorstore = retrieve_from_vector_db(vector_db_path=vector_db_path, embedding=embedding_wrapped)

#type(retriever),type(vectorstore)

### Prompt

In [17]:
# maps natural-language user terms

FILTER_MAP = {
    # parties, matching df_to_document_batches
    "spd":       ("party", "SPD"),
    "cdu/csu":   ("party", "CDU/CSU"),
    "cdu":       ("party", "CDU"),
    "csu":       ("party", "CSU"),
    "grünen":    ("party", "GRÜNE"),
    "grüne":     ("party", "GRÜNE"),
    "fdp":       ("party", "FDP"),
    "linke":     ("party", "LINKE"),
    "afd":       ("party", "AfD"),
    "fraktionslos": ("party", "fraktionslos"),
    "parteilos": ("party", "parteilos"),
    # -- add other parties to complete list since 1949
    
    # government status, values are int 0/1 as stored in FAISS
    "kabinett":         ("party", "Cabinet"),
    "regierungspartei": ("government_status", 1),
    "bundesregierung":  ("government_status", 1),
    "regierung":        ("government_status", 1),
    "opposition":       ("government_status", 0),
    
    # roles 
    "kanzler":         ("role", "Bundeskanzler"),
    "bundeskanzler":   ("role", "Bundeskanzler"),
    "kanzlerin":       ("role", "Bundeskanzler"),
    "bundeskanzlerin": ("role", "Bundeskanzler"),
    "minister":        ("role", "Bundesminister"),
    "bundesminister":  ("role", "Bundesminister"),
    "staatssekretär":  ("role", "Staatssekretär"),
    "staatsminister":  ("role", "Staatsminister"),
    "abgeordnete":     ("role", "MdB"),
    "mdb":             ("role", "MdB"),
    "mitglied des bundestags": ("role", "MdB"),
    
    # legislative periods 
    "19. wahlperiode": ("legislative_period", 19),
    "19 wahlperiode":  ("legislative_period", 19),
    "wp19":            ("legislative_period", 19),
    "18. wahlperiode": ("legislative_period", 18),
    "18 wahlperiode":  ("legislative_period", 18),
    "wp18":            ("legislative_period", 18),
    # -- add remaining lps from 1-17 for full data set
    
    # time frame 
    "2021": ("year", "2021"),
    "2020": ("year", "2020"),
    "2019": ("year", "2019"),
    "2018": ("year", "2018"),
    "2017": ("year", "2017"),
    # -- add remaining time frames til 1949 for full data set
}

In [31]:
# extract metadata filters and semantic query from free-text user input 
# handles: party, role, government_status, cabinet, speaker, session, date, legislative_period
# returns (semantic_query, filters) tuple
def parse_query_filters(user_input: str) -> tuple[str, dict]:
    """
    Parses a free-text user query into a semantic search string and a
    metadata filter dict for FAISS.

    Extraction order (all additive, multiple filters supported):
      1. Party affiliation     — matched via FILTER_MAP
      2. Government status     — Regierungspartei / Opposition
      3. Role                  — Minister, Staatssekretär, Abgeordnete
      4. Legislative period    — "19. Wahlperiode", "WP19", etc.
      5. Session               — "Sitzung 42", "42. Sitzung"
      6. Date                  — ISO (2019-03-14) or German (14.03.2019)
      7. Speaker name          — "von [Name]" or "Redner [Name]"

    Returns:
        semantic       : str   — residual text after filter terms removed
        filters        : dict  — metadata key/value pairs for FAISS filter
    """
    filters = {}
    semantic = user_input.lower()

    # 1–4: FILTER_MAP lookup
    for term, (key, value) in sorted(FILTER_MAP.items(), key=lambda x: len(x[0]), reverse=True):
        pattern = r'(?<!\w)' + re.escape(term) + r'(?!\w)'
        if re.search(pattern, semantic):
            filters[key] = value
            semantic = re.sub(pattern, '', semantic)
            
    # party filter takes precendence over government_status        
    if "party" in filters and "government_status" in filters:
        del filters["government_status"]

    # 5: Session — "Sitzung 42" or "42. Sitzung"
    session_match = re.search(r'(\d+)\.\s*sitzung|sitzung\s*(\d+)', semantic)
    if session_match:
        session_num = session_match.group(1) or session_match.group(2)
        filters["session"] = session_num
        semantic = re.sub(r'(\d+)\.\s*sitzung|sitzung\s*(\d+)', '', semantic)

    # 6: Date — ISO (2019-03-14) or German (14.03.2019)
    date_match = re.search(r'\d{4}-\d{2}-\d{2}|\d{2}\.\d{2}\.\d{4}', semantic)
    if date_match:
        filters["date"] = date_match.group()
        semantic = semantic.replace(date_match.group(), "")

    # 7: Speaker — capture full name (Vorname + Nachname) anywhere in query
    # strip possessive suffix for matching (e.g. "Merkels" → "Merkel")
    semantic_for_speaker = re.sub(r'(\w)s\b', r'\1', semantic)
    
    speaker_found = None
    for name in sorted(KNOWN_SPEAKERS, key=len, reverse=True):
        if name.lower() in semantic_for_speaker:
            speaker_found = name
            break

    if speaker_found:
        filters["speaker_name"] = speaker_found
        # remove from original semantic (both with and without possessive)
        semantic = re.sub(re.escape(name.lower()) + r's?\b', '', semantic)

    semantic = re.sub(r'\s+', ' ', semantic).strip()
    return semantic, filters

In [32]:
# inject chunk metadata into context string (-> LLM)
# attributes extracted arguments to specific speakers/ sessions
def format_context_with_metadata(docs: list[Document]) -> str:
    """
    Prepends a metadata header to each retrieved chunk so the LLM can
    attribute extracted arguments to specific speakers, parties and sessions.
    Fields with missing/NaN values are omitted from the header.
    """
    GOV_STATUS_MAP = {1: "Regierungspartei", 0: "Opposition"}
    
    formatted = []
    for doc in docs:
        m = doc.metadata
        header_parts = []
        # headers as defined in df_to_document_batches()
        if m.get("speaker_name"):       header_parts.append(f"Redner: {m['speaker_name']}")
        if m.get("party"):              header_parts.append(f"Partei: {m['party']}")
        if "government_status" in m:    header_parts.append(f"Regierungsstatus: {GOV_STATUS_MAP.get(m['government_status'], m['government_status'])}")
        if m.get("role"):               header_parts.append(f"Rolle: {m['role']}")
        if m.get("date"):               header_parts.append(f"Datum: {m['date']}")
        if m.get("year"):               header_parts.append(f"Jahr: {m['year']}")
        if m.get("session"):            header_parts.append(f"Sitzung: {m['session']}")
        if m.get("legislative_period"): header_parts.append(f"Wahlperiode: {m['legislative_period']}")
        
        header = f"[{' | '.join(header_parts)}]" if header_parts else ""
        formatted.append(f"{header}\n{doc.page_content}".strip())
        
    return "\n\n---\n\n".join(formatted)

In [33]:
# retriever with dynamic metadata filters. Falls back to unfiltered retriever if no filters detected
def get_filtered_retriever(vectorstore, filters: dict, k: int = 15):
    """
    Returns a retriever with metadata filters applied.
    Falls back to unfiltered retrieval if filters dict is empty.
    """
    search_kwargs = {"k": k}
    if filters:
        search_kwargs["filter"] = filters
        if "speaker_name" in filters:
            search_kwargs["k"] = 30
            search_kwargs["fetch_k"] = 500
        elif "party" in filters:
            search_kwargs["k"] = 50
            search_kwargs["fetch_k"] = 300
        elif "government_status" in filters:
            search_kwargs["k"] = 30
            search_kwargs["fetch_k"] = 200
                
    return vectorstore.as_retriever(search_kwargs=search_kwargs)

In [34]:
prompt = ChatPromptTemplate.from_template(
    """
    Du bist ein Assistent für Argumentationsanalyse, spezialisiert auf parlamentarische Debatten des Deutschen Bundestags.

    Deine Aufgabe ist es, die Argumentationsstruktur aus dem untenstehenden Redeauszug zu extrahieren,
    wobei du die Frage des Nutzers/ der Nutzerin als thematischen Fokus verwendest.
    
    Du erhältst einen Redeausschnitt aus einem Bundestagsprotokoll (Kontext) 
    und eine Frage zur Position des Sprechers, einer Partei oder der Regierung / Opposition. Extrahiere das zentrale Argument 
    im vorgegebenen JSON-Format. Erfinde nichts; halte dich ausschließlich 
    an den gegebenen Kontext.

    Definitionen:
    - claim: die zentrale politische Position oder der Vorschlag, für den oder gegen den argumentiert wird
    - grounds: faktische oder normative Belege, die den claim stützen
    - rebuttal: antizipierter Einwand der Gegenseite + Widerlegung durch den Sprecher
    - attack: offensive Kritik des Sprechers an der Gegenposition

    ### Beispiel 1:
    Frage 1: "Wie begründet Heiko Maas die Notwendigkeit, die bestehende Rüstungskontrollarchitektur zu erhalten?"

    Kontext 1: [Redner: Heiko Maas | Partei: SPD | Rolle: Bundesminister des Auswärtigen | Datum: 2018-04-19 | Sitzung: 26 | Wahlperiode: 19 | Regierungsstatus: Regierungspartei]

    Russland hat die konventionelle Rüstungskontrolle in Europa seit über zehn Jahren einseitig suspendiert. Die USA und Russland werfen sich gegenseitig vor, den INF-Vertrag zu verletzen. Gleichzeitig rüsten China, Indien und Pakistan erheblich auf. Kaum eines der Regime, die jahrzehntelang zu unserer Sicherheit beigetragen haben, funktioniert noch vollständig. Auf deutsches Betreiben hin sind die Verhandlungen über ein Verbot waffenfähigen Spaltmaterials nähergerückt.

    Output 1:
    {{
    "claim": "Die bestehende Rüstungskontrollarchitektur muss aktiv erhalten und modernisiert werden, da ihre Erosion die Sicherheit in Europa direkt gefährdet.",
    "grounds": ["Russland hat die Rüstungskontrolle seit über zehn Jahren einseitig suspendiert; China, Indien und Pakistan rüsten erheblich auf; kaum ein Sicherheitsregime funktioniert noch vollständig."],
    "rebuttal": ["Trotz schwieriger Lage zeigen Fortschritte bei den Verhandlungen über waffenfähiges Spaltmaterial, dass Abrüstungsschritte auch im aktuellen Klima möglich sind."],
    "attack": ["Wer die regelbasierte Abrüstungsarchitektur aufgibt, überlässt das Feld denjenigen, die auf Aufrüstung setzen."],
    "speaker": "Heiko Maas",
    "party": "SPD",
    "government_status": "Regierungspartei",
    "role": "Bundesminister des Auswärtigen",
    "date": "2018-04-19",
    "session": "26",
    "legislative_period": "19",
    "confidence": "high",
    "note": null
    }}

    ### Beispiel 2:
    Frage 2: "Welche Position vertritt Gregor Gysi zum Atomwaffenverbotsvertrag?"

    Kontext 2: [Redner: Gregor Gysi | Partei: DIE LINKE | Rolle: MdB | Datum: 2021-01-29 | Sitzung: 207 | Wahlperiode: 19 | Regierungsstatus: Opposition]

    122 Staaten der UNO haben den Atomwaffenverbotsvertrag beschlossen. Deutschland muss alle Maßnahmen zum Abbau des Atomwaffenarsenals unterstützen. Doch die Bundesregierung steht abseits. Das Hauptargument der Regierung ist ein angeblicher Widerspruch zum Atomwaffensperrvertrag. Der Wissenschaftliche Dienst des Bundestages hat widerlegt, dass der Nichtverbreitungsvertrag durch den Verbotsvertrag angegriffen wird. Nachdem alle Argumente der Regierung widerlegt sind, muss ich feststellen: CDU/CSU und SPD wollen kein Verbot von Atomwaffen.

    Output 2:
    {{
    "claim": "Deutschland muss dem Atomwaffenverbotsvertrag beitreten – aus historischer Verantwortung und weil alle Gegenargumente der Bundesregierung sachlich widerlegt sind.",
    "grounds": ["122 Staaten haben den Vertrag beschlossen; der Wissenschaftliche Dienst des Bundestages bestätigt, dass er den Nichtverbreitungsvertrag ergänzt, ohne ihn zu untergraben."],
    "rebuttal": ["Die Bundesregierung behauptet einen Widerspruch zum Nichtverbreitungsvertrag. Der Wissenschaftliche Dienst hat diesen Einwand explizit widerlegt."],
    "attack": ["CDU/CSU und SPD wollen kein Verbot von Atomwaffen – sie ignorieren die Faktenlage und die historische Verantwortung Deutschlands."],
    "speaker": "Gregor Gysi",
    "party": "DIE LINKE",
    "government_status": "Opposition",
    "role": "MdB",
    "date": "2021-01-29",
    "session": "207",
    "legislative_period": "19",
    "confidence": "high",
    "note": null
    }}

    ### Beispiel 3:
    Frage 3: "Wie argumentiert die CDU/CSU für die Rolle der Landwirtschaft im Klimaschutz?"

    Kontext 3: [Redner: Gitta Connemann | Partei: CDU/CSU | Rolle: MdB | Datum: 2020-09-17 | Sitzung: 176 | Wahlperiode: 19 | Regierungsstatus: Regierungspartei]

    Die größten Klimaschützer in Deutschland sind unsere Waldbauern. Jeder Hektar Wald bindet 8 Tonnen CO2 pro Jahr. Selbsternannte Experten fordern den Verzicht auf die Waldbewirtschaftung. Das ist verantwortungslos, denn Holz aus Ländern mit niedrigeren Standards müsste importiert werden. Grünland ohne Bewirtschaftung verbuscht – Artenvielfalt adieu. Ohne Nutzung keine Nachhaltigkeit.

    Output 3:
    {{
    "claim": "Nachhaltigkeit erfordert die aktive Bewirtschaftung durch Landwirte und Waldbauern – ohne sie gibt es weder Klima- noch Artenschutz.",
    "grounds": ["Jeder Hektar Wald bindet 8 Tonnen CO2 pro Jahr; artenreiches Grünland ist durch landwirtschaftliche Nutzung über Generationen entstanden."],
    "rebuttal": ["Der Verzicht auf Waldbewirtschaftung ist verantwortungslos, denn Holz aus Ländern mit niedrigeren Standards müsste importiert werden – das bricht die Nachhaltigkeit."],
    "attack": ["Grünlandnutzung zu verbieten bewirkt das Gegenteil: Ohne Bewirtschaftung verbuscht Grünland und die Artenvielfalt geht verloren."],
    "speaker": "Gitta Connemann",
    "party": "CDU/CSU",
    "government_status": "Regierungspartei",
    "role": "MdB",
    "date": "2020-09-17",
    "session": "176",
    "legislative_period": "19",
    "confidence": "high",
    "note": null
    }}

    ### Beispiel 4:
    Frage 4: "Wie kritisiert die FDP die Mehrwertsteuersenkung im Zweiten Corona-Steuerhilfegesetz?"

    Kontext 4: [Redner: Christian Dürr | Partei: FDP | Rolle: MdB | Datum: 2020-06-29 | Sitzung: 168 | Wahlperiode: 19 | Regierungsstatus: Opposition]

    Die vorübergehende Mehrwertsteuersenkung ist nicht die richtige Antwort auf diese Krise. Sie ist verbunden mit absurdem bürokratischem Aufwand. Selbst im optimistischsten Fall sind es 30 Euro für einen durchschnittlichen Haushalt pro Monat. Die Profiteure werden Onlineversandhändler wie Amazon sein. Wir legen konkrete Alternativen vor: Abschaffung des Solidaritätszuschlages, Abschaffung des Mittelstandsbauchs und eine negative Gewinnsteuer in Höhe von 25 Milliarden Euro. Ihre eigenen Experten sagen: Die Mehrwertsteuersenkung ist ineffizient und bürokratisch.

    Output 4:
    {{
    "claim": "Die vorübergehende Mehrwertsteuersenkung ist die falsche Krisenmaßnahme – sie belastet den Mittelstand bürokratisch und kommt bei den Bürgern kaum an.",
    "grounds": ["Die Mehrwertsteuersenkung bringt selbst im optimistischsten Fall nur 30 Euro pro Haushalt und Monat; selbst die von der Koalition benannten Experten bewerten sie als ineffizient."],
    "rebuttal": ["Drei konkrete Alternativen: vollständige Abschaffung des Solidaritätszuschlages, Abschaffung des Mittelstandsbauchs und eine negative Gewinnsteuer über steuerliche Verlustverrechnung in Höhe von 25 Milliarden Euro."],
    "attack": ["Die Profiteure der Mehrwertsteuersenkung sind nicht die Krisenbetroffenen, sondern Onlineversandhändler wie Amazon – der Mittelstand wird mit Bürokratie belastet, Amazon wird entlastet."],
    "speaker": "Christian Dürr",
    "party": "FDP",
    "government_status": "Opposition",
    "role": "MdB",
    "date": "2020-06-29",
    "session": "168",
    "legislative_period": "19",
    "confidence": "high",
    "note": null
    }}

    ### Beispiel 5:
    Frage 5: "Wie argumentiert die AfD gegen die Gleichwertigkeit von digitalem Unterricht und Präsenzunterricht?"

    Kontext 5: [Redner: Nicole Höchst | Partei: AfD | Rolle: MdB | Datum: 2020-10-07 | Sitzung: 182 | Wahlperiode: 19 | Regierungsstatus: Opposition]

    Die unterstellte Gleichwertigkeit von digitalem Unterricht und Präsenzunterricht geht völlig an der Lebenswirklichkeit vorbei. Eine Forsa-Umfrage bestätigt: Digitaler Unterricht kann Präsenzunterricht nicht ersetzen, ist dem Lernerfolg nicht dienlich und familienorganisatorisch untauglich. Die staatliche Kinderbetreuung hat komplett versagt. Die klassische Familie mit traditioneller Aufgabenverteilung ist pandemiekrisensicher.

    Output 5:
    {{
    "claim": "Präsenzunterricht muss auch in Krisenzeiten sichergestellt sein – digitaler Unterricht ist kein gleichwertiger Ersatz, und die staatliche Kinderbetreuung hat in der Pandemie versagt.",
    "grounds": ["Eine Forsa-Umfrage bestätigt: Digitaler Unterricht kann Präsenzunterricht nicht ersetzen, ist dem Lernerfolg nicht dienlich und familienorganisatorisch untauglich."],
    "rebuttal": ["Die Gleichwertigkeit von digitalem und Präsenzunterricht geht an der Lebenswirklichkeit von Schülern, Lehrern und Eltern vorbei und ist durch empirische Befunde widerlegt."],
    "attack": ["Die übrigen Parteien halten die Fremdbetreuung von Kindern für vordringlich förderungswürdig und degradieren damit Eltern, die ihre Kinder zu Hause betreuen möchten."],
    "speaker": "Nicole Höchst",
    "party": "AfD",
    "government_status": "Opposition",
    "role": "MdB",
    "date": "2020-10-07",
    "session": "182",
    "legislative_period": "19",
    "confidence": "high",
    "note": null
    }}

    ### Beispiel 6:
    Frage 6: "Wie argumentieren die Grünen für eine Verschärfung der Mietpreisbremse?"

    Kontext 6: [Redner: Daniela Wagner | Partei: GRÜNE | Rolle: MdB | Datum: 2018-03-01 | Sitzung: 17 | Wahlperiode: 19 | Regierungsstatus: Opposition]

    Die Mietpreisbremse hat viele Ausnahmen und Umgehungsmöglichkeiten. Die Folge waren noch schneller steigende Preise. Allein von 2016 auf 2017 sind die Mietpreise um bis zu 10 Prozent gestiegen. Armutsgefährdete Haushalte müssen bis knapp die Hälfte ihres Einkommens für die Miete aufwenden. Die Losung „Bauen, bauen, bauen" hat nichts geändert. Wir brauchen die Pflicht zur Angabe der Vormiete, die Abschaffung der Rügepflicht und eine Absenkung der Modernisierungsumlage.

    Output 6:
    {{
    "claim": "Die Mietpreisbremse muss zu einem wirksamen Instrument gemacht werden – in der aktuellen Form ist sie wirkungslos, weil zu viele Ausnahmen bestehen.",
    "grounds": ["Allein von 2016 auf 2017 sind die Mietpreise um bis zu 10 Prozent gestiegen; armutsgefährdete Haushalte müssen bis knapp die Hälfte ihres Einkommens für die Miete aufwenden."],
    "rebuttal": ["Die Wohnungswirtschaft fordert die Abschaffung der Mietpreisbremse. Das Gegenteil ist nötig: Pflicht zur Angabe der Vormiete, Abschaffung der Rügepflicht und Absenkung der Modernisierungsumlage."],
    "attack": ["Die Losung 'Bauen, bauen, bauen' hat nichts geändert – steuerliche Anreize allein produzieren Mitnahmeeffekte, schaffen aber keinen bezahlbaren Wohnraum."],
    "speaker": "Daniela Wagner",
    "party": "GRÜNE",
    "government_status": "Opposition",
    "role": "MdB",
    "date": "2018-03-01",
    "session": "17",
    "legislative_period": "19",
    "confidence": "high",
    "note": null
    }}

    Frage:
    {question}
        
    Regeln:
    - Verwende NUR Informationen aus dem Kontext, keine externen Kenntnisse
    - Erfinde NIEMALS Metadaten. Wenn ein Feld nicht im Kontext-Header steht, setze es auf null.
    - Wenn Metadaten nicht im Kontext-Header stehen, setze das Feld auf null — schreibe NIEMALS erklärende Texte wie "(kein expliziter Redner)" in Metadatenfelder
    - Zitiere oder paraphrasiere grounds, rebuttal und attack eng aus dem Text, nichts erfinden
    - Claim darf als knappe Synthese formuliert werden
    - Fokussiere die Extraktion auf die Frage des Nutzers, ignoriere thematisch irrelevante Textteile
    - Entnimm IMMER Redner/in, Partei, Rolle, Datum, Sitzung, Regierungspartei/ Opposition und Wahlperiode aus den Metadaten-Kopfzeilen im Kontext
    - Setze Felder auf null, wenn die entsprechenden Informationen im Kontext nicht vorhanden sind
    - Wenn kein klarer claim erkennbar ist, setze "claim" auf null
    - Wenn keine grounds vorhanden sind, gib eine leere Liste zurück
    - Wenn keine rebuttals oder attacks vorhanden sind, gib eine leere Liste zurück
    - Gib NUR valides JSON zurück, keine Erklärung, keine Einleitung, keine Markdown-Formatierung
    - Wenn der Kontext keine relevanten Informationen zur Anfrage enthält, gib folgendes zurück:
        {{"claim": null, "grounds": [], "rebuttal": [], "attack": [], 
        "confidence": "low", "note": "Keine relevanten Informationen gefunden. 
        Versuche die Suchanfrage zu vereinfachen oder Filter zu reduzieren."}}

    WICHTIG: Deine Antwort muss ausschließlich mit {{ beginnen. Kein Text davor. Deine Antwort muss mit }} abschließen. Kein Text danach.
    
    Kontext:
    {context}
    
    """)

In [35]:
# pydantic schema for output
# core argument based on toulmin's model of argumentation
class ArgumentStructure(BaseModel):
    # core argument
    claim:               Optional[str] = None
    grounds:             Optional[list[str]] = []
    rebuttal:            Optional[list[str]] = []   # conditions under which claim does not hold
    attack:              Optional[list[str]] = []
    # speaker attribution
    speaker:             Optional[str]
    party:               Optional[str]
    government_status:   Optional[str]
    role:                Optional[str]
    date:                Optional[str]
    session:             Optional[str]
    legislative_period:  Optional[str]
    # quality
    confidence:          Literal["high", "medium", "low"] 
    reasoning:           Optional[str] = None
    note:                Optional[str] = None # carries message if no result
    



In [36]:
# check and format LLM output: according to Pydantic schema
def parse_llm_output(raw_output: str) -> dict:
    """
    Cleans, parses and validates LLM output against ArgumentStructure schema.
    Returns {"status": "ok", "data": ...} or error dict with raw output for debugging.
    """
    try:
        # strip markdown fences
        cleaned = re.sub(r"```json|```", "", raw_output).strip()
        
        # extract JSON block even if model adds prose before/after
        json_match = re.search(r"\{.*\}", cleaned, re.DOTALL)
        if not json_match:
            # try to repair truncated JSON by appending closing brace
            if cleaned.strip().startswith("{"):
                cleaned_repaired = cleaned.strip() + "}"
                json_match = re.search(r"\{.*\}", cleaned_repaired, re.DOTALL)
            if not json_match:
                return {"status": "json_error", "raw": raw_output, "error": "No JSON block found in output"}
        
        json_str = json_match.group()
        parsed = json.loads(json_str)
        validated = ArgumentStructure(**parsed)
        return {"status": "ok", "data": validated.model_dump()}

    except json.JSONDecodeError as e:
        return {"status": "json_error", "raw": raw_output, "error": str(e)}
    except ValidationError as e:
        return {"status": "validation_error", "raw": raw_output, "error": str(e)}


In [44]:
# runnable RAG pipeline (LCEL)
# parse user input -> retrieve
def connect_chains(retriever, vectorstore):
    """
    Full RAG chain with dynamic metadata filtering.
    Connects retriever, prompt and llm into a RAG chain for argument extraction.
    User question both drives semantic search and focuses the extraction.

    Flow:
      1. Parse user input → (semantic_query, filters)
      2. Build filtered retriever from vectorstore
      3. Retrieve top-k chunks matching semantic query + metadata filters
      4. Format chunks with metadata headers
      5. Pass formatted context + original question to prompt
      6. LLM generates structured argument JSON
      7. Parse and validate output against ArgumentStructure schema
    """
    def retrieve_and_format(user_input: str) -> dict:
        semantic_query, filters = parse_query_filters(user_input) #extract metadata filters and semantic query from user input
        filtered_retriever = get_filtered_retriever(vectorstore, filters) #retriever applying meta data filters
        docs = filtered_retriever.invoke(semantic_query)
        return {
            "context":  format_context_with_metadata(docs),
            "question": user_input   # original question — preserves full user intent for LLM
        }

    rag_chain = (
        RunnableLambda(retrieve_and_format)
        | prompt
        | llm
        | RunnableLambda(lambda msg: parse_llm_output(msg.content))
    )
    return rag_chain

In [45]:
# pass vectorstore as second argument
rag_chain = connect_chains(retriever, vectorstore)

In [46]:
# attributed labels for chat output
ATTRIBUTION_FIELDS = [
    ("speaker",            "👤 Redner"),
    ("party",              "🏛️ Partei"),
    ("role",               "💼 Rolle"),
    ("government_status",   "📍 Kabinett/ Opposition"),
    ("date",               "📅 Datum"),
    ("session",            "📋 Sitzung"),
    ("legislative_period", "🗳️ Wahlperiode"),
]


In [47]:
# rag chain            
def chat_with_rag(chain):
    """
    Interactive RAG chat loop with argument mining output.
    Handles: structured output, no-results case, JSON errors.
    """
    print("Willkommen im ChatBundestag 🏛️. Gib eine Frage zu den Bundestagsdebatten ein.")
    print("Schreibe 'exit', um den Chat zu beenden.\n")

    while True:
        user_input = input("Du: ").strip()

        if not user_input:
            continue # skip accidental empty enters to prevent crash

        if user_input.lower() in ["exit", "quit"]:
            print("Chat wird beendet. Auf Wiedersehen!")
            break

        try:
            result = chain.invoke(user_input)

            # ── JSON / validation error ──────────────────────────────────────
            if result["status"] != "ok":
                print(f"\n⚠️  Parsing fehlgeschlagen ({result['status']})")
                print(f"Fehler: {result['error']}")
                print(f"Rohausgabe:\n{result['raw']}\n") 
                continue

            data = result["data"]

            # ── No relevant results ──────────────────────────────────────────
            if data.get("note"):          
                print(f"\n⚠️  {data['note']}\n") 
                continue

            if not data["claim"]:
                print("\n⚠️  Keine relevanten Informationen gefunden.")
                print("Tipp: Versuche die Anfrage zu vereinfachen oder weniger Filter zu verwenden.\n")
                continue

            # ── Normal structured output ─────────────────────────────────────
            print("\n" + "─" * 60)

            # Speaker / context info
            speaker   = data.get("speaker")   or "unbekannt"
            party     = data.get("party")      or "?"
            government_status = data.get("government_status") or "?"
            role      = data.get("role")       or "?"
            date      = data.get("date")       or "?"
            period    = data.get("legislative_period") or "?"
            session   = data.get("session") or "?"
            confidence = data.get("confidence") or "?"
            
            print(f"👤  {speaker} ({party} · {role} · {government_status})")
            print(f"📅  {date}  |  Legislaturperiode: {period} | Sitzung: {session} ")
            print(f"🎯  Konfidenz: {confidence}")
            print("─" * 60)

            # Claim
            print(f"\n📌  Claim:\n    {data['claim']}\n")

            all_arguments = (data.get("grounds") or []) + (data.get("rebuttal") or []) + (data.get("attack") or []) #get value, if None: fall back to empty list
            if all_arguments:
                print("💬  Argumente:")
                for a in all_arguments:
                    print(f"    • {a}")
            else:
                print("💬  Argumente: keine gefunden")

            print("─" * 60 + "\n")

        except Exception as e:
            print(f"\n❌  Fehler: {e}\n")
            


In [48]:
# run chat
chat_with_rag(rag_chain)

Willkommen im ChatBundestag 🏛️. Gib eine Frage zu den Bundestagsdebatten ein.
Schreibe 'exit', um den Chat zu beenden.



Du:  Wie argumentieren die Grünen für mehr digitale Bildung?



❌  Fehler: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kh3jayqge7pak5txdmvbp5t2` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 98698, Requested 7411. Please try again in 1h27m58.175999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}



KeyboardInterrupt: Interrupted by user

### error log
- don't generate output if query is unclear/ ill-formulated / lacking essential components
- how is the topic introduced for which to retrieve claim/premises ? Query should filter the database preemptively - so query + context in prompt!
- MPs can be MGs, as well (check data!)
- "query" vs query: sometimes different

### erroneous query log
- Wie kritisieren die Grünen die außenpolitischen Ambitionen der Bundesregierung? (gives output with Grünen speaker but argument from system prompt AfD)
- Welche Argumente werden gegen die Energiewende genannt? (same speaker as 'für die Energiewende'. Correct?)
- Was sind die Argumente der AfD für oder gegen das Klimaschutzgesetz?
- Wie steht der Finanzminister zur Mietpreisbremse? (hallucinated date, same arguments as for: "Wie steht der Wirtschaftsminister zur Mietpreisbremse?)"
#
- Wie argumentieren die Grünen für mehr digitale Bildung?
- Welche Position vertritt die Linke zum Klimaschutzgesetz?

### no-info-found log
- Wie steht die FDP zur Mietpreisbremse?
- Wie steht die Bundesregierung zur Mietpreisbremse?
- Wie steht die Regierung zur Mietpreisbremse?
- Wie steht die CDU zum Brexit? 
- Was sind Angela Merkels Argumente für für Europäische Integration?
- Was sind die Argumente der Grünen für das Klimaschutzgesetz? 
- Finde Merkels Argumente zur Maskenaffäre 
- Welche Position vertritt Svenja Schulze zum Klimaschutzgesetz?
- Welche Position vertritt Svenja Schulze zur CO2-Bepreisung ?
- Wie steht Angela Merkel zum Thema Maskenpflicht? 
- Was sind die Argumente der CDU/CSU für den Brexit?
- Welche Position vertritt die Bundesregierung zum Brexit?
#
- Welche Position vertritt Dietmar Bartsch zum Klimaschutzgesetz?


### expected query log
- Welche Position vertritt die Bundesregierung zur nuklearen Abrüstung? 
- Welche Position vertritt Gregor Gysi zur europäischen Verteidigungspolitik?
- Welche Position vertritt die Linke zur nuklearen Abrüstung? 
- Welche Position vertritt Gregor Gysi zum Atomwaffenverbotsvertrag?
- Welche Argumente werden für die Energiewende genannt?
- Welche Position vertritt die Linke zur nuklearen Abrüstung?
- Wie steht die AfD zum Klimaschutzgesetz?
- Welche Position vertritt Claudia Roth zum Klimaschutz?


### query types (wie steht/ welche Position vertritt / welche Argument für o. gegen gibt / wie kritisiert ... )
#### prio1
- Wie steht [name] zu [topic]?
- Wie steht [government_status=Bundesregierung] zu [topic]?
- Wie steht [party] zu [topic]?

#### prio2
- Wie steht [role] zu [topic] ? (role other than MdB; here: MGs)
- Welche Argumente werden für [topic] genannt? (gegen?)

#### prio3
- Wie steht [government_status=Opposition] zu [topic]? (one output per [party in government_status=opposition])

### known errors
- Cabinet inconsistency: This does create a functional issue, but a narrow one. If someone asks "Wie steht die CDU/CSU zum Klimaschutz?", the filter becomes {"party": "CDU/CSU"}. Any CDU minister whose party value is "Cabinet" instead of "CDU/CSU" will be filtered out — so you'd miss Merkel, Scheuer, Altmaier, etc. for party-based queries. They'd only appear via role-based queries ("Bundeskanzlerin") or government-status queries ("Bundesregierung").
- True filter-then-search. FAISS doesn't support that natively. Switch to Qdrant/Weaviate (future work). LangChain doesn't filter first, then search.